# Lab 00 — Data & Chat Templates (★☆☆ · CORE)

**Watch first:** [Lecture 0](https://rlhfbook.com/course/) + [Lecture 1](https://rlhfbook.com/course/) · [Chapter 3](https://rlhfbook.com/)

**Goal:** know, position by position, what a causal-LM loss would learn from a chat-formatted example — and be able to explain every token. CPU only; no training.

**Answer key (open only after your version passes):** `rlhf-book/code/instruction_tuning/utils.py` lines 153–244 (`_encode_batch`, `_collate`).

## Setup

CPU-only. `transformers` (tokenizer use only), `datasets`, `numpy`. No torch required for anything in this lab.

In [ ]:
# TODO(00.setup): imports + seed; keep this lab torch-free
import numpy as np

from starter import (
    FIXED_PANEL_PROMPTS,
    IGNORE_INDEX,
    build_prompt_masked_labels,
    collate_right_pad,
    decode_unmasked_only,
    diff_tokenizations,
    generation_comparison,
    inspect_tokenizer,
    length_stats,
    load_no_robots_sample,
    make_triples,
    pooling_corruption_demo,
    tokenize_chat,
    tokenize_plain,
)

print('FIXED_PANEL_PROMPTS:', FIXED_PANEL_PROMPTS)
print('IGNORE_INDEX =', IGNORE_INDEX)

## Part 1 — Tokenizers: raw vs chat-templated (assignment item 1)

Load the three lab tokenizers (`Qwen/Qwen3-0.6B-Base`, `allenai/OLMo-2-0425-1B` + template donor `allenai/OLMo-2-0425-1B-SFT`, `HuggingFaceTB/SmolLM2-360M-Instruct`) and render their special-token surfaces.

In [ ]:
# TODO(00.a): load the three tokenizers (AutoTokenizer); for OLMo pass the -SFT
# donor via tokenizer.chat_template so the base model still renders a template.
# qwen_tok  = ...  # Qwen/Qwen3-0.6B-Base
# olmo_tok  = ...  # allenai/OLMo-2-0425-1B (+ donor template)
# smol_tok  = ...  # HuggingFaceTB/SmolLM2-360M-Instruct

# TODO: print inspect_tokenizer(tok) for each; note who has a pad token and
# who does not (this matters in Part 4).

In [ ]:
# TODO(00.a): pick ONE 2-turn conversation from no_robots[:50] and diff renders.
# messages = [{'role': 'user', 'content': ...}, {'role': 'assistant', 'content': ...}]
# raw_text  = your chosen join formatting (record it in a comment!)
# plain_ids = tokenize_plain(qwen_tok, raw_text)
# chat_ids  = tokenize_chat(qwen_tok, messages)
# print(diff_tokenizations(plain_ids, chat_ids, qwen_tok))
# TODO: decode extra_prefix_tokens / extra_suffix_tokens and name each template token.

## Part 2 — Prompt-masked labels (assignment item 2, the core)

Implement `build_prompt_masked_labels` so that **only the final assistant turn** carries gradient. Then verify by decoding only unmasked positions.

Decisions to defend in a markdown cell:
- Is the assistant turn's *header* token supervised? (README §9 — check the answer key afterwards.)
- Is the post-answer EOS supervised? (Why must it be?)

In [ ]:
# TODO(00.b): mask a conversation that ends in an assistant turn.
# input_ids, labels = build_prompt_masked_labels(qwen_tok, messages, 512)
# assert len(input_ids) == len(labels)
# TODO: verify the two invariants by hand:
#   1. labels[i] == IGNORE_INDEX for every prompt-side position
#   2. decode_unmasked_only(qwen_tok, input_ids, labels) returns EXACTLY the
#      final assistant turn (answer + stop token), nothing else

In [ ]:
# TODO(00.b): the 'done when' artifact — print the (token, id, label) triples.
# triples = make_triples(qwen_tok, input_ids, labels)
# TODO: print all rows and add a markdown cell explaining EVERY row
# (system/user tokens masked? header supervised? EOS? truncation tail?)

## Part 3 — Experiments

1. **Decode-unmasked ×10** on the first 10 usable no_robots rows.
2. **Length distributions** raw vs templated, base vs instruct.

In [ ]:
# TODO(00.e): load_no_robots_sample(50) — needs network; run in Colab or with a
# local HF cache. Filter to conversations ending in an assistant turn and say why.
# TODO(00.expt1): for 10 samples, check decode_unmasked_only recovers exactly
# the final assistant turn; record any failure and diagnose it.

In [ ]:
# TODO(00.d): length_stats on raw vs templated token counts per tokenizer.
# Collect lists of lengths, call length_stats(...), and tabulate base vs instruct.
# TODO: write 3 bullets on WHY templated renders are longer (headers, turn marks, EOS).

## Part 4 — Batching: right-padding & pooling corruption (assignment item 4)

In [ ]:
# TODO(00.c): build 3-4 variable-length (input_ids, labels) examples from Part 2,
# batch = collate_right_pad(examples, pad_token_id)
# TODO: assert rectangular shapes; attention_mask 0 / labels -100 exactly at pads.
# demo = pooling_corruption_demo(batch)
# TODO: write the paragraph explaining what a naive mean/max pool over the full
# padded length would learn from pads (and what LEFT padding changes).

## Part 5 — Base continuation vs instruct answer-and-stop (assignment item 3)

In [ ]:
# TODO(00.d): load small CPU models (e.g. SmolLM2-360M base vs -Instruct, or the
# 0.6B pair) and fill the 6-prompt panel:
# base_out = generation_comparison(base_model, base_tok, FIXED_PANEL_PROMPTS, 'base')
# inst_out = generation_comparison(inst_model, inst_tok, FIXED_PANEL_PROMPTS, 'instruct')
# TODO: print side-by-side + one-line commentary per prompt (base rambles on?);
# connect 'never stops' to the Part 2 question: what happens if EOS is not supervised?

## Wrap-up

- [ ] `(token, id, label)` triples printed, every row explained.
- [ ] Decode-unmasked check green on 10 samples.
- [ ] 6-prompt base-vs-instruct panel saved with commentary.
- [ ] Answers written to README §8's three questions.
- [ ] `cd labs/00_data_chat_templates && python3 -m pytest tests/ -q` passes.

**Only now** open the answer key — `rlhf-book/code/instruction_tuning/utils.py:153-244` — and diff your boundary alignment + collate against it. Explain every difference in writing.